In [18]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_RAW  = Path("data/raw")
DATA_PROC = Path("data/processed")

print("Libraries loaded ✅")
print(f"Raw data folder   : {DATA_RAW}")
print(f"Processed folder  : {DATA_PROC}")

Libraries loaded ✅
Raw data folder   : data\raw
Processed folder  : data\processed


In [19]:
listings = pd.read_csv(DATA_RAW / "listings.csv", low_memory=False)

print(f"Shape: {listings.shape}")
print(f"Rows: {listings.shape[0]:,}")
print(f"Columns: {listings.shape[1]}")
print("\nFirst 5 columns:")
print(listings.head())

Shape: (8274, 79)
Rows: 8,274
Columns: 79

First 5 columns:
       id                          listing_url       scrape_id last_scraped  \
0   97945   https://www.airbnb.com/rooms/97945  20250927041758   2025-09-27   
1  114695  https://www.airbnb.com/rooms/114695  20250927041758   2025-09-27   
2  127383  https://www.airbnb.com/rooms/127383  20250927041758   2025-09-27   
3  159634  https://www.airbnb.com/rooms/159634  20250927041758   2025-09-27   
4  170154  https://www.airbnb.com/rooms/170154  20250927041758   2025-09-27   

            source                                               name  \
0      city scrape                   Deluxw-Apartm. with roof terrace   
1  previous scrape                 Apartment Munich/East with sundeck   
2      city scrape                  City apartment next to Pinakothek   
3  previous scrape  Fancy, bright central roof top flat and homeof...   
4      city scrape              Own floor & bath, parking & breakfast   

                          

In [20]:
missing = listings.isnull().sum()
missing_pct = (missing / len(listings) * 100).round(2)

missing_df = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
}).sort_values("missing_pct", ascending=False)

# Show only columns that have missing values
missing_df = missing_df[missing_df["missing_count"] > 0]

print(f"Columns with missing values: {len(missing_df)}")
print(missing_df)

Columns with missing values: 43
                              missing_count  missing_pct
neighbourhood_group_cleansed           8274       100.00
calendar_updated                       8274       100.00
license                                8236        99.54
host_neighbourhood                     6497        78.52
neighborhood_overview                  5862        70.85
neighbourhood                          5862        70.85
host_about                             5178        62.58
estimated_revenue_l365d                2671        32.28
price                                  2671        32.28
beds                                   2658        32.12
bathrooms                              2658        32.12
host_response_rate                     2404        29.05
host_response_time                     2404        29.05
host_location                          2014        24.34
review_scores_cleanliness              1916        23.16
review_scores_location                 1916        23.16

In [21]:
# Drop columns that are not useful
cols_to_drop = [
    "neighbourhood_group_cleansed",  
    "calendar_updated",              
    "license",                       
    "host_neighbourhood",            
    "neighborhood_overview",         
    "neighbourhood",                 
    "host_about",                    
    "estimated_revenue_l365d",       
]

listings.drop(columns=cols_to_drop, inplace=True)

print(f"Columns remaining: {listings.shape[1]}")
print("✅ Useless columns dropped!")

Columns remaining: 71
✅ Useless columns dropped!


In [22]:
#  convert to a number
listings["price"] = (listings["price"]
                     .astype(str)
                     .str.replace("$", "", regex=False)
                     .str.replace(",", "", regex=False)
                     .pipe(pd.to_numeric, errors="coerce"))

# Drop rows where price is missing
listings = listings.dropna(subset=["price"])

# Remove price outliers (bottom 1% and top 1%)
Q1 = listings["price"].quantile(0.01)
Q3 = listings["price"].quantile(0.99)
listings = listings[listings["price"].between(Q1, Q3)]

print(f"Rows remaining after price cleaning: {listings.shape[0]:,}")
print(f"\nPrice stats:")
print(listings["price"].describe().round(2))

Rows remaining after price cleaning: 5,490

Price stats:
count    5490.00
mean      238.77
std       190.25
min        41.00
25%       113.00
50%       179.00
75%       296.00
max      1229.00
Name: price, dtype: float64


In [23]:
review_cols = [
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value"
]

for col in review_cols:
    median = listings[col].median()
    listings[col] = listings[col].fillna(median)
    print(f"  {col:<35} median={median:.2f}")

print("\n✅ Review scores cleaned!")

  review_scores_rating                median=4.89
  review_scores_accuracy              median=4.91
  review_scores_cleanliness           median=4.88
  review_scores_checkin               median=4.95
  review_scores_communication         median=4.98
  review_scores_location              median=4.88
  review_scores_value                 median=4.71

✅ Review scores cleaned!


In [24]:
# Fill missing bedrooms and beds with median
listings["bedrooms"] = listings["bedrooms"].fillna(listings["bedrooms"].median())
listings["beds"] = listings["beds"].fillna(listings["beds"].median())
listings["bathrooms"] = listings["bathrooms"].fillna(listings["bathrooms"].median())

# Fill missing host_is_superhost with "f" (assume not superhost)
listings["host_is_superhost"] = listings["host_is_superhost"].fillna("f")

# Fill missing host_response_rate with "unknown"
listings["host_response_rate"] = listings["host_response_rate"].fillna("unknown")

# Drop rows where host_since is missing (only 4 rows)
listings = listings.dropna(subset=["host_since"])

print(f"Rows remaining: {listings.shape[0]:,}")
print(f"Columns remaining: {listings.shape[1]}")

# Final missing check
remaining_missing = listings.isnull().sum().sum()
print(f"Total missing values remaining: {remaining_missing}")
print("✅ Columns cleaned!")

Rows remaining: 5,487
Columns remaining: 71
Total missing values remaining: 6912
✅ Columns cleaned!


In [25]:
missing = listings.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

host_location             1444
reviews_per_month         1365
first_review              1365
last_review               1365
host_response_time         636
host_acceptance_rate       334
description                225
has_availability           163
bathrooms_text               3
maximum_maximum_nights       3
minimum_maximum_nights       3
maximum_minimum_nights       3
minimum_minimum_nights       3
dtype: int64


In [26]:
listings.drop(columns=cols_to_drop, inplace=True)

# Fill reviews_per_month, first_review, last_review with unknown/0
listings["reviews_per_month"] = listings["reviews_per_month"].fillna(0)
listings["first_review"] = listings["first_review"].fillna("unknown")
listings["last_review"] = listings["last_review"].fillna("unknown")

# Final check
remaining_missing = listings.isnull().sum().sum()
print(f"Rows: {listings.shape[0]:,}")
print(f"Columns: {listings.shape[1]}")
print(f"Total missing values remaining: {remaining_missing}")
print("✅ Transform complete!")

KeyError: "['neighbourhood_group_cleansed', 'calendar_updated', 'license', 'host_neighbourhood', 'neighborhood_overview', 'neighbourhood', 'host_about', 'estimated_revenue_l365d'] not found in axis"

In [27]:
# Drop columns that are not useful
cols_to_drop = [
    "host_location",
    "host_response_time",
    "host_acceptance_rate",
    "description",
    "has_availability",
    "bathrooms_text",
    "maximum_maximum_nights",
    "minimum_maximum_nights",
    "maximum_minimum_nights",
    "minimum_minimum_nights",
]

# Only drop columns that still exist
cols_to_drop = [c for c in cols_to_drop if c in listings.columns]

listings.drop(columns=cols_to_drop, inplace=True)

# Fill remaining missing values
listings["reviews_per_month"] = listings["reviews_per_month"].fillna(0)
listings["first_review"] = listings["first_review"].fillna("unknown")
listings["last_review"] = listings["last_review"].fillna("unknown")

# Final check
remaining_missing = listings.isnull().sum().sum()
print(f"Rows: {listings.shape[0]:,}")
print(f"Columns: {listings.shape[1]}")
print(f"Total missing values remaining: {remaining_missing}")
print("✅ Transform complete!")

Rows: 5,487
Columns: 61
Total missing values remaining: 0
✅ Transform complete!


In [28]:
DATA_PROC = Path("data/processed")

listings.to_csv(DATA_PROC / "listings_clean.csv", index=False)

print(f"✅ Saved: listings_clean.csv")
print(f"Rows: {listings.shape[0]:,}")
print(f"Columns: {listings.shape[1]}")

✅ Saved: listings_clean.csv
Rows: 5,487
Columns: 61
